# 📝 ReAct·멀티툴 에이전트 과제 LV2 정답 (강사용)

각 문제의 **모범답안 + 해설**입니다. 학생이 스스로 푼 뒤 비교하도록 안내하세요.

- 경로는 정답 노트북 기준 `../../day20_ReAct_멀티툴_에이전트/data/` 입니다.
- 검색·계산처럼 모델과 무관하게 정해지는 것은 **값을 정확히** 검사합니다.
- 모델은 **실제로 호출**되므로 답 문장·단계 수·도구 호출 횟수는 실행마다 달라집니다. 라우팅·검색 문제의 자가채점은 **도구 자체**(계산값·검색 결과)만 보고, 에이전트가 그 도구를 부르는지는 답안 셀의 `tool_names(...)` **출력으로 확인**합니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../../day20_ReAct_멀티툴_에이전트/.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 오늘 쓸 모델. 18일차에서 배운 그대로입니다(이 셀은 실행만 하세요).
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [ ]:
# [제공 코드] 구청 민원 도구 재료. 수수료·처리기한 표와 공통 import (이 셀은 실행만 하세요)
import re
import datetime
import pandas as pd
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.agents import create_agent
_fees = pd.read_csv('../../day20_ReAct_멀티툴_에이전트/data/fees.csv')
FEE_TABLE = dict(zip(_fees['document'], _fees['fee']))
_dls = pd.read_csv('../../day20_ReAct_멀티툴_에이전트/data/deadlines.csv')
DEADLINE_TABLE = dict(zip(_dls['minwon'], _dls['days']))

def tool_names(result):
    """메시지 기록에서 실제로 불린 도구 이름 목록을 뽑는다."""
    names = []
    for message in result['messages']:
        if isinstance(message, AIMessage):   # 도구 호출은 AIMessage 에만 담긴다
            for call in message.tool_calls:
                names.append(call['name'])
    return names
print('준비 완료: 수수료·처리기한 표, tool_names()')

## 1. 검색한 조각에 앞뒤 문맥을 붙여 돌려주는 도구
**배경**: 긴 안내문은 조각으로 잘라 색인합니다. 그래야 질문과 맞는 대목이 정확히 잡히니까요. 그런데 조각 하나만 모델에게 넘기면 **문장이 잘린 채**로 갑니다. 그래서 검색은 조각으로 하고, **넘길 때는 그 앞뒤 조각까지 이어 붙입니다.**

구청 민원 **긴 안내문 6편**을 조각낸 벡터DB 가 이미 만들어져 있습니다(한 편이 4조각, 모두 24조각). 아래 제공 셀은 그것을 **열기만** 합니다. 조각의 꼬리표에는 `doc_id`·`chunk_no`·`chunk_total`·`title` 이 들어 있습니다.

**요구사항**: `@tool` 로 **`search_guide(query: str) -> str`** 를 만드세요.

- `guide_retriever.invoke(query)` 로 조각을 찾습니다(상위 2개).
- 찾은 조각마다 **앞뒤 한 개씩**(자기 자신 포함 최대 3개)의 조각 번호를 고릅니다. **0 보다 작거나 `chunk_total` 이상인 번호는 버립니다.**
- 그 번호의 조각을 **같은 문서에서** 꺼내 옵니다 - `guide_store.get(where=...)` 를 쓰고, 조건은 `doc_id` 가 같고 `chunk_no` 가 고른 번호 안에 드는 것입니다.
- **꺼낸 순서는 보장되지 않습니다.** `chunk_no` 로 다시 정렬해야 글이 이어집니다.
- 조각마다 앞에 **`[제목] `** 을 붙이고(꼬리표의 `title`), 결과를 하나의 문자열로 이어 돌려줍니다.
- docstring 에 **언제 쓰는 도구인지**를 적으세요.

**예시**: `search_guide.invoke({'query': '지방세를 늦게 내면 어떻게 되나요?'})` 의 결과에는 가산금을 설명한 조각(**`삼 퍼센트`**)뿐 아니라 그 **앞 조각**(`납부는 은행 창구…`)과 **뒤 조각**(`환급`)의 내용도 함께 들어 있습니다.

> `where` 에 조건을 둘 이상 걸 때는 `{'$and': [{...}, {...}]}` 로 묶습니다. 값이 같은지는 `{'$eq': 값}`, 목록 안에 드는지는 `{'$in': [번호들]}` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 검색 -> 번호 고르기 -> 그 번호의 조각 꺼내기 -> 번호로 정렬 -> 이어 붙이기, 다섯 단계다.

세부구현:
1. 검색 결과를 하나씩 돌면서 그 조각의 꼬리표를 꺼낸다.
2. 자기 번호를 가운데 두고 앞뒤로 하나씩 벌린 범위를 만든 뒤, 문서 밖 번호를 걸러 낸다.
3. 저장소의 get 으로 같은 doc_id 이면서 그 번호들에 드는 조각을 꺼낸다.
   3-1. 결과에는 꼬리표 목록과 본문 목록이 따로 들어 있다 - 둘을 짝지어 다룬다.
4. 짝지은 것을 조각 번호 기준으로 정렬한 뒤 본문만 줄바꿈으로 잇는다.
5. 앞에 제목을 붙여 블록을 만들고, 블록들을 빈 줄로 이어 반환한다.
```

</details>

In [ ]:
# [제공 코드] 미리 만들어 둔 벡터DB 를 엽니다 - 색인은 이미 끝나 있습니다(이 셀은 실행만 하세요).
#  임베딩 모델은 '질문을 벡터로 바꾸는 데' 필요해 한 번 올립니다(처음 한 번은 잠시 걸립니다).
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

_embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')
guide_store = Chroma(persist_directory='../../day20_ReAct_멀티툴_에이전트/data/chroma_day20',
                     collection_name='minwon_guide_lv2',
                     embedding_function=_embeddings)
guide_retriever = guide_store.as_retriever(search_kwargs={'k': 2})

print('민원 안내문 조각 수:', len(guide_store.get()['ids']))

In [ ]:
# 찾는 단위는 작은 조각, 넘기는 단위는 조각+앞뒤 — 목적이 다르니 단위도 다르다
@tool
def search_guide(query: str) -> str:
    """구청 민원 안내문에서 질문과 관련된 대목을 앞뒤 문맥까지 붙여 돌려준다.

    전입신고·여권·건축 허가·영업 신고·지방세·대형 폐기물 같은 민원 절차를 물을 때 쓴다.
    """
    blocks = []
    for hit in guide_retriever.invoke(query):
        tag = hit.metadata
        want = [n for n in range(tag['chunk_no'] - 1, tag['chunk_no'] + 2)
                if 0 <= n < tag['chunk_total']]
        got = guide_store.get(where={'$and': [{'doc_id': {'$eq': tag['doc_id']}},
                                              {'chunk_no': {'$in': want}}]})
        # get 은 순서를 보장하지 않는다 - 번호로 다시 줄을 세워야 글이 이어진다
        pairs = sorted(zip(got['metadatas'], got['documents']), key=lambda p: p[0]['chunk_no'])
        blocks.append(f"[{tag['title']}] " + '\n'.join(text for _tag, text in pairs))
    return '\n\n'.join(blocks)


print(search_guide.invoke({'query': '지방세를 늦게 내면 어떻게 되나요?'})[:300])

In [ ]:
# [자가채점] - 검색은 임베딩 계산이라 결정적입니다. 앞뒤가 실제로 붙었는지까지 봅니다.
from langchain_core.tools import BaseTool
assert isinstance(search_guide, BaseTool), '@tool 을 붙였는지 확인하세요'
assert search_guide.description.strip(), 'docstring 을 적었는지 확인하세요'
_out = search_guide.invoke({'query': '지방세를 늦게 내면 어떻게 되나요?'})
assert isinstance(_out, str), '문자열을 돌려줘야 합니다'
assert '[지방세 납부 안내]' in _out, '조각 앞에 [제목] 을 붙였는지 확인하세요'
assert '삼 퍼센트' in _out, '검색이 집은 조각이 들어 있어야 합니다'
assert '납부는 은행 창구' in _out, '앞 조각이 빠졌습니다'
assert '환급' in _out, '뒤 조각이 빠졌습니다'
# 문서 경계를 넘지 않았는지 - 지방세 질문에 다른 안내문 제목이 섞여 나오면 안 됩니다.
assert '[여권 발급 안내]' not in _out
print('✅ 통과!')

**해설**: **검색 단위와 전달 단위를 다르게** 두는 문제입니다. 조각이 작아야 질문과 맞는 대목이 정확히 잡히고(크면 관계없는 글이 섞여 순위가 흐려집니다), 모델이 읽을 때는 앞뒤가 있어야 문장이 이어집니다.

**함정 둘**. **1)** `get` 이 돌려주는 순서는 **보장되지 않습니다** — 정렬하지 않으면 뒷조각이 앞에 붙어 글이 뒤엉킵니다(에러가 안 나서 더 위험합니다). **2)** 번호를 고를 때 `0 <= n < chunk_total` 로 막지 않으면 **없는 번호**를 달라고 하게 되고, 그 자리는 조용히 비어 버립니다.

**`window` 는 앞뒤로 몇 조각을 더 붙일지 정하는 파라미터입니다.** 넓히면 문맥이 넉넉해지는 대신 프롬프트가 길어지고 관계없는 글이 섞입니다. 여기서는 앞뒤 한 개로 고정했습니다.

## 2. 멀티툴 에이전트로 수수료 질문 라우팅하기
**배경**: 수수료 도구와 처리기한 도구를 **함께** 붙인 에이전트를 만들어, 질문에 맞는 도구가 불리는지 봅니다.

**요구사항**:
- `@tool` 로 **`calc_fee(document, count)`**(수수료=단가×매수, `FEE_TABLE`, `document`.strip)와 **`lookup_deadline(minwon)`**(처리일수 `'N일'`, `DEADLINE_TABLE`, `minwon`.strip)를 만드세요. docstring 을 각각 분명히 적습니다.
- 두 도구로 에이전트 **`civil_agent`** 를 만들고(system_prompt 자유), **'인감증명서 3장 수수료는?'** 을 물어 결과를 `r_fee` 에 담으세요.

**예시**: 이 질문에는 `calc_fee` 가 불려야 합니다(`tool_names(r_fee)` 에 `'calc_fee'` 포함).

<details><summary>힌트</summary>

```text
접근방법:
- 두 도구를 @tool 로 만들고 create_agent 에 리스트로 함께 넘긴다. 수수료 질문을 invoke 한다.

세부구현:
1. calc_fee: FEE_TABLE.get(document.strip(), 0) * count 를 문자열로 반환(docstring: 발급 수수료 계산).
2. lookup_deadline: DEADLINE_TABLE.get(minwon.strip()) 로 일수를 찾아 'N일'/'모름'(docstring: 처리기한).
3. civil_agent = create_agent(model, [calc_fee, lookup_deadline], system_prompt=...).
4. civil_agent.invoke 로 예시 질문을 넣어 r_fee 에 담는다.
```

</details>

In [ ]:
# 두 도구의 docstring 이 서로 다른 쓰임을 분명히 말해 줘야 에이전트가 골라 쓸 수 있다
@tool
def calc_fee(document: str, count: int) -> str:
    """민원 서류 이름과 매수를 받아 총 발급 수수료(원)를 계산한다."""
    return str(FEE_TABLE.get(document.strip(), 0) * count)

@tool
def lookup_deadline(minwon: str) -> str:
    """민원 이름을 받아 접수 후 처리에 걸리는 기간(며칠)을 알려 준다."""
    days = DEADLINE_TABLE.get(minwon.strip())
    return f'{days}일' if days is not None else '모름'

civil_agent = create_agent(model, [calc_fee, lookup_deadline],
                           system_prompt='너는 구청 민원 도우미다. 질문에 맞는 도구를 골라 답하라.')
# 도구가 둘이어도 수수료 질문에는 하나만 불린다 — 아래 출력으로 확인한다
r_fee = civil_agent.invoke({'messages': [HumanMessage('인감증명서 3장 수수료는?')]})
print(tool_names(r_fee), '|', r_fee['messages'][-1].text)

In [ ]:
# [자가채점] - 도구 자체(계산값)를 검사. 라우팅은 위 답안 셀의 tool_names(r_fee) 출력으로 확인.
from langchain_core.tools import BaseTool
assert isinstance(calc_fee, BaseTool) and isinstance(lookup_deadline, BaseTool)   # @tool 로 만든 도구인가
assert calc_fee.invoke({'document': '인감증명서', 'count': 3}) == '1800'   # 도구는 결정적
assert calc_fee.name == 'calc_fee' and lookup_deadline.name == 'lookup_deadline'
# civil_agent 를 실제로 돌렸는지 - 기록에 질문과 '모델이 만든 AIMessage' 가 함께 있어야 합니다.
assert any(m.content == '인감증명서 3장 수수료는?' for m in r_fee['messages'])
assert any(isinstance(m, AIMessage) and m.response_metadata
           for m in r_fee['messages'])
print('✅ 통과!')

**해설**: 두 도구를 붙여도 에이전트는 **수수료 질문에 수수료 도구**를 고릅니다. 도구 값은 결정적(1800)이라 자가채점하고, 라우팅(어떤 도구가 불렸는지)은 답안 셀의 `tool_names(r_fee)` 출력으로 확인합니다 — 보통 `['calc_fee']` 가 찍힙니다(모델이 정하는 일이라 호출 횟수·순서는 실행마다 다를 수 있습니다).

## 3. 멀티툴 에이전트로 처리기한 질문 라우팅하기
**배경**: 2번과 **같은 `civil_agent`** 에 이번엔 **처리기한** 질문을 넣습니다(같은 라우팅 개념, 다른 방향).

**요구사항**:
- 2번의 `civil_agent` 에 **'건축 허가는 처리에 며칠 걸리나요?'** 를 물어 결과를 `r_dl` 에 담으세요.

**예시**: 이 질문에는 `lookup_deadline` 이 불려야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 이미 만든 civil_agent 를 그대로 쓴다. 처리기한 질문만 invoke 한다.

세부구현:
1. civil_agent.invoke 로 처리기한 질문을 넣어 r_dl 에 담는다.
2. tool_names(r_dl) 를 출력해 확인한다.
```

</details>

In [ ]:
# 같은 에이전트에 방향이 다른 질문 — 이번엔 처리기한 도구가 불려야 한다
r_dl = civil_agent.invoke({'messages': [HumanMessage('건축 허가는 처리에 며칠 걸리나요?')]})
print(tool_names(r_dl), '|', r_dl['messages'][-1].text)

In [ ]:
# [자가채점] - 처리기한 도구 값은 결정적으로 검사. 라우팅은 위 tool_names(r_dl) 출력으로 확인.
assert lookup_deadline.invoke({'minwon': '건축 허가'}) == '14일'   # 도구 계산은 결정적
# 그 질문으로 civil_agent 를 실제로 돌렸는지 확인합니다 - 손으로 만든 메시지 목록에는
# 모델이 만든 AIMessage 가 없어 여기서 걸립니다.
assert any(m.content == '건축 허가는 처리에 며칠 걸리나요?' for m in r_dl['messages'])
assert any(isinstance(m, AIMessage) and m.response_metadata
           for m in r_dl['messages'])
print('✅ 통과!')

**해설**: 같은 에이전트가 질문의 **의도에 따라 다른 도구**를 고릅니다(2번은 수수료, 3번은 처리기한). 자가채점은 처리기한 도구 값(`'14일'`)을 결정적으로 검사하고, 실제 라우팅은 답안 셀의 `tool_names(r_dl)` 출력으로 확인합니다 — 보통 `['lookup_deadline']` 입니다.

---
## 4. 필요할 때만 검색하는 RAG Agent
**배경**: 민원 FAQ 를 검색하는 도구를 붙여, **민원 질문에는 검색하고 인사에는 검색하지 않는** 에이전트를 만듭니다(교안의 필요시-검색을 다른 도메인으로).

**요구사항**:
- 아래 제공 셀이 만든 `retrieve_minwon(query, k)` 검색기를 **`@tool` 로 감싸** `search_minwon(query)` 을 만드세요(docstring: 구청 민원 안내를 검색). 입력은 `.strip()`, 상위 2개를 줄바꿈으로 이어 반환.
- 이 도구로 에이전트 **`faq_agent`** 를 만들고, **'전입신고는 어떻게 하나요?'**(→`q_faq`)와 **'안녕하세요'**(→`q_hi`) 두 질문을 각각 invoke 하세요.

**예시**: `tool_names(q_faq)` 에는 `'search_minwon'` 이 있고, `tool_names(q_hi)` 는 비어 있어야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- retrieve_minwon 을 @tool 로 감싸고, create_agent 에 붙여 두 질문을 각각 invoke 한다.

세부구현:
1. @tool 로 search_minwon(query) 정의 — retrieve_minwon(query.strip(), 2) 를 줄바꿈 연결해 반환.
2. faq_agent: create_agent 에 그 도구 하나를 리스트로 넘기고, system_prompt 로 '민원 질문에만 검색 도구를 쓰고 인사엔 그냥 답하라'는 역할을 준다.
3. q_faq, q_hi 에 각각 invoke 결과를 담는다.
```

</details>

In [ ]:
# [제공 코드] 검색기 준비. 15~16일차에 배운 벡터 검색을 함수 하나로 묶어 둡니다.
# (불러오는 데 잠시 걸립니다. 이 단원의 주제는 이 검색을 '도구'로 감싸 에이전트에 붙이는 것입니다.)
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer


def build_retriever(csv_path, text_columns, collection_name):
    """CSV 를 임베딩해 벡터DB에 넣고, 그 코퍼스를 검색하는 함수를 돌려준다(15~16일차 절차 그대로)."""
    embedder = SentenceTransformer('jhgan/ko-sroberta-multitask')
    df = pd.read_csv(csv_path)
    texts = df[text_columns].agg(' '.join, axis=1).tolist()
    client = chromadb.EphemeralClient()
    # 같은 이름이 남아 있으면 지우고 새로 만든다(셀을 다시 실행해도 안전하게).
    if collection_name in [c.name for c in client.list_collections()]:
        client.delete_collection(collection_name)
    collection = client.create_collection(
        collection_name, metadata={'hnsw:space': 'cosine'})
    collection.add(ids=df['id'].tolist(),
                   embeddings=embedder.encode(texts, normalize_embeddings=True).tolist(),
                   documents=texts)

    def retrieve(query, k=2):
        """질문과 의미가 가장 가까운 문서 본문 k개를 리스트로 돌려준다."""
        hits = collection.query(
            query_embeddings=embedder.encode([query], normalize_embeddings=True).tolist(),
            n_results=k)
        return hits['documents'][0]

    return retrieve


retrieve_minwon = build_retriever('../../day20_ReAct_멀티툴_에이전트/data/minwon_faq.csv', ['category', 'text'], 'minwon_faq')
print('검색기 준비 완료: retrieve_minwon')

In [ ]:
# 검색기를 도구로 감싸면 '언제 검색할지' 판단을 에이전트에 넘길 수 있다
@tool
def search_minwon(query: str) -> str:
    """구청 민원 안내(전입신고·증명서 발급·여권 등)에서 관련 내용을 찾아 돌려준다."""
    return '\n'.join(retrieve_minwon(query.strip(), 2))

faq_agent = create_agent(model, [search_minwon],
                         system_prompt='너는 구청 민원 도우미다. 민원 질문에만 검색 도구를 쓰고 인사엔 그냥 답하라.')
q_faq = faq_agent.invoke({'messages': [HumanMessage('전입신고는 어떻게 하나요?')]})
# 인사에는 도구가 안 불려 [] 가 나오는 것이 이 문제의 핵심이다
q_hi = faq_agent.invoke({'messages': [HumanMessage('안녕하세요')]})
print('민원:', tool_names(q_faq), '| 인사:', tool_names(q_hi))

In [ ]:
# [자가채점] - 검색 도구 자체(결정적)를 검사. 필요할 때만 검색하는 라우팅은 위 출력으로 확인.
from langchain_core.tools import BaseTool
assert isinstance(search_minwon, BaseTool)   # @tool 로 감쌌는가
assert search_minwon.name == 'search_minwon'
assert search_minwon.invoke({'query': '전입신고'}).strip()   # 검색은 결정적 - 관련 안내를 찾아 돌려준다
# 두 질문을 실제로 돌렸는지 확인합니다(어떤 도구를 골랐는지는 채점하지 않습니다).
# 모델이 만든 AIMessage 가 있어야 하므로, 손으로 만든 메시지 목록으로는 통과할 수 없습니다.
for one in (q_faq, q_hi):
    assert any(isinstance(m, AIMessage) and m.response_metadata
           for m in one['messages'])
assert any(m.content == '전입신고는 어떻게 하나요?' for m in q_faq['messages'])
assert any(m.content == '안녕하세요' for m in q_hi['messages'])
print('✅ 통과!')

**해설**: 검색을 **도구**로 주면 에이전트가 필요를 판단합니다 — 민원 질문엔 검색하고 인사엔 부르지 않습니다. 그 라우팅은 답안 셀의 `민원:`/`인사:` 출력으로 확인하고(보통 `민원: ['search_minwon'] | 인사: []`), 자가채점은 검색 도구 자체가 관련 안내를 찾아 오는지(결정적)를 봅니다. 검색은 내 컴퓨터에서 도는 임베딩 계산이라 모델 호출과 무관하게 늘 같은 결과를 줍니다 — 그래서 이건 값으로 채점할 수 있습니다.

## 5. 검색과 계산이 섞인 질문 처리하기
**배경**: 실제 질문은 **검색과 계산이 섞이기도** 합니다. 검색 도구와 수수료 도구를 함께 붙이면 에이전트가 필요한 도구를 (하나 또는 여럿) 고릅니다.

**요구사항**:
- `search_minwon`(4번)과 `calc_fee`(2번) 두 도구로 에이전트 **`mix_agent`** 를 만드세요.
- **'주민등록등본 2장 발급 수수료는 얼마인가요?'** 를 물어 결과를 `r_mix` 에 담으세요.

**예시**: 이 질문은 수수료 계산이 필요하므로 `tool_names(r_mix)` 에 **`'calc_fee'`** 가 포함돼야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 도구를 함께 붙인 에이전트를 만들고 수수료가 필요한 질문을 invoke 한다.

세부구현:
1. mix_agent = create_agent(model, [search_minwon, calc_fee], system_prompt=...).
2. 예시 질문을 invoke 해 r_mix 에 담는다.
3. tool_names(r_mix) 를 확인한다.
```

</details>

In [ ]:
# 도구가 여럿이어도 질문이 요구하는 것만 불린다(검색을 곁들일 수도 있다)
mix_agent = create_agent(model, [search_minwon, calc_fee],
                         system_prompt='너는 구청 민원 도우미다. 질문에 맞는 도구를 골라 답하라.')
r_mix = mix_agent.invoke({'messages': [HumanMessage('주민등록등본 2장 발급 수수료는 얼마인가요?')]})
print(tool_names(r_mix), '|', r_mix['messages'][-1].text)

In [ ]:
# [자가채점] - 수수료 도구 값은 결정적으로 검사. 혼합 질문에서 무엇이 불렸는지는 위 출력으로 확인.
assert calc_fee.invoke({'document': '주민등록등본', 'count': 2}) == '800'   # 도구 계산은 결정적
# mix_agent 로 그 질문을 실제로 돌렸는지까지 확인합니다(모델이 만든 AIMessage 가 있어야 합니다).
assert any(m.content == '주민등록등본 2장 발급 수수료는 얼마인가요?' for m in r_mix['messages'])
assert any(isinstance(m, AIMessage) and m.response_metadata
           for m in r_mix['messages'])
print('✅ 통과!')

**해설**: 여러 도구가 있어도 에이전트는 **질문이 요구하는 도구**를 고릅니다 — 이 질문은 수수료 계산이 핵심이라 `calc_fee` 가 불립니다(검색도 함께 부를 수 있음). 자가채점은 수수료 도구 값(800)을 결정적으로 검사하고, 실제로 무엇이 불렸는지는 답안 셀의 `tool_names(r_mix)` 출력으로 확인합니다.

## 6. 실패 수리: 도구는 불렸는데 결과가 없다
**배경**: 아래 도구는 **옳게 불리는데도** 답이 "없음" 으로 나옵니다. 에러도 안 납니다. 교안 5절에서 본 **조용한 실패**입니다. 기록의 **인자**를 보고 원인을 짚어 고치세요.

```python
@tool
def deadline_raw(minwon: str) -> str:
    """민원 처리기한을 돌려준다."""
    return str(DEADLINE_TABLE.get(minwon, '없음'))
```

실제로 나온 기록입니다 — 도구는 맞게 불렸습니다.

```text
질문: 여권 발급은 며칠 걸리나요?
불린 도구: ['deadline_raw']
넘어간 인자: [{'minwon': '여권 발급은'}]
결과: 없음
```

**요구사항**: 함수가 아니라 **도구**를 만듭니다.

- `@tool` 을 붙인 **`deadline_fixed(minwon: str) -> str`** 를 정의하세요.
- 받은 이름의 **앞뒤 공백을 떼고**, 끝에 붙은 **조사 한 글자**(`은`·`는`·`이`·`가`·`의`)도 떼어 낸 뒤 `DEADLINE_TABLE` 에서 찾으세요.
- 찾으면 **`'{민원} 처리기한: {일수}일'`** 형식으로 돌려주세요.
- 못 찾으면 `'없음'` 이 아니라 **고를 수 있는 민원 이름을 알려 주는 안내 문자열**을 돌려주세요 (그 문장에 `DEADLINE_TABLE` 의 키들이 들어가야 합니다).
- docstring 에는 *언제 쓰는 도구인지*를 적으세요.

**예시**: `deadline_fixed.invoke({'minwon': '여권 발급은'})` 은 `'여권 발급 처리기한: 8일'` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모델이 넘기는 표면형을 우리가 통제할 수 없으니, 도구가 열쇠 모양으로 맞춘다.

세부구현:
1. 앞뒤 공백을 떼어 낸다.
2. 끝에 붙은 조사 한 글자를 떼어 낸다 (문자열 메서드로 여러 후보를 한 번에 뗄 수 있다).
3. 표에 있으면 지정된 형식으로 만든다.
4. 없으면 고를 수 있는 이름들을 이어 붙인 안내 문장을 돌려준다.
```

</details>

In [ ]:
@tool
def deadline_fixed(minwon: str) -> str:
    """민원 이름으로 처리기한(일수)을 알려준다.

    여권 발급·건축 허가처럼 구청에 접수하는 민원이 며칠 걸리는지 물을 때 쓴다.
    """
    # 사람은 '여권 발급은' 처럼 조사를 붙여 말한다 - 우리 열쇠는 조사가 없으므로 도구가 다듬는다.
    name = minwon.strip().rstrip('은는이가의')
    if name not in DEADLINE_TABLE:
        return f"'{minwon}' 은 없는 민원입니다. 가능한 민원: {', '.join(DEADLINE_TABLE)}"
    return f'{name} 처리기한: {DEADLINE_TABLE[name]}일'

In [ ]:
# [자가채점]
# 도구는 모델을 거치지 않고 직접 부를 수 있다 - 값을 정확히 검사한다.
_want = f"여권 발급 처리기한: {DEADLINE_TABLE['여권 발급']}일"
for surface in ['여권 발급', '여권 발급은', ' 여권 발급 ']:
    assert deadline_fixed.invoke({'minwon': surface}) == _want, f'{surface!r} 를 못 다듬었습니다'
# 못 찾았을 때 '없음' 만 돌려주면 조용한 실패가 그대로 남는다.
_miss = deadline_fixed.invoke({'minwon': '우주여권 발급'})
assert _miss != '없음', '없는 민원에는 안내 문자열을 돌려주세요'
assert any(k in _miss for k in DEADLINE_TABLE), '안내에 고를 수 있는 민원 이름이 있어야 합니다'
print('✅ 통과!')

**해설**: 이 실패가 무서운 이유는 **아무 신호도 없다**는 데 있습니다. 도구는 옳게 불렸고, 에러도 없고, 값도 돌아왔습니다. 그저 틀렸을 뿐이지요. 그래서 도구 결과가 비어 보일 때는 **기록의 인자**를 먼저 봅니다 — 무엇이 열쇠로 쓰였는지가 거기 있습니다.

**`rstrip` 이 여러 글자를 받는 이유**: `rstrip('은는이가의')` 는 그 **집합에 속한 문자**를 끝에서부터 계속 떼어 냅니다. 하나의 문자열을 떼는 것이 아닙니다. 그래서 조사 후보를 한 번에 처리할 수 있습니다.

**흔한 실수**: 없는 값에 `'없음'` 이나 `0` 을 돌려주는 것입니다. 모델은 그것을 사실로 읽고 사용자에게 그대로 전합니다. 안내 문자열이면 모델이 "민원 이름을 다시 확인해 달라" 처럼 쓸 수 있습니다.

## 7. 실패 수리: 같은 도구를 자꾸 부른다
**배경**: 6번과 **다른 층위**의 고장입니다. 아래 도구는 값도 맞고 인자도 잘 오는데, 에이전트가 **같은 도구를 여러 번** 부릅니다. 기록을 보고 원인을 짚어 고치세요.

```python
@tool
def fee_repeat(document: str) -> str:
    """서류 수수료를 알려준다. 값이 바뀔 수 있으니 확실히 하려면 여러 번 확인하는 것이 좋다."""
    return str(FEE_TABLE.get(document.strip(), 0))
```

```text
질문: 주민등록등본 수수료가 얼마인가요?
불린 도구: ['fee_repeat', 'fee_repeat', 'fee_repeat']
```

**요구사항**: `@tool` 을 붙인 **`fee_once(document: str) -> str`** 를 정의하세요.

- docstring 에서 **반복을 권하는 문장을 빼고**, 결과가 **확정**이라는 것과 *언제 쓰는 도구인지*를 적으세요.
- 서류 이름의 **앞뒤 공백을 떼고** `FEE_TABLE` 에서 단가를 찾으세요.
- 찾으면 **`'{서류} 수수료: {금액}원 (확인 완료)'`** 형식으로 돌려주세요.
- 못 찾으면 고를 수 있는 서류 이름을 알려 주는 **안내 문자열**을 돌려주세요.

**예시**: `fee_once.invoke({'document': '주민등록등본'})` 은 `'주민등록등본 수수료: 400원 (확인 완료)'` 입니다(금액은 표의 실제 값).

<details><summary>힌트</summary>

```text
접근방법:
- docstring 은 모델에게 주는 지시문이다. 반복을 권하면 정말로 반복한다.

세부구현:
1. 설명에서 여러 번 부르라는 문장을 뺀다.
2. 결과가 확정이라 다시 부를 필요가 없다는 문장을 넣는다.
3. 서류 이름의 공백을 떼고 표에서 단가를 찾는다.
4. 없으면 고를 수 있는 이름들을 이어 붙인 안내 문장을 돌려준다.
```

</details>

In [ ]:
@tool
def fee_once(document: str) -> str:
    """서류 이름으로 발급 수수료를 한 번 조회한다. 결과는 확정이므로 다시 부를 필요가 없다.

    주민등록등본·가족관계증명서처럼 구청에서 떼는 서류의 수수료를 물을 때 쓴다.
    """
    name = document.strip()
    if name not in FEE_TABLE:
        return f"'{name}' 은 발급 목록에 없습니다. 가능한 서류: {', '.join(FEE_TABLE)}"
    return f'{name} 수수료: {FEE_TABLE[name]}원 (확인 완료)'

In [ ]:
# [자가채점]
_want = f"주민등록등본 수수료: {FEE_TABLE['주민등록등본']}원 (확인 완료)"
assert fee_once.invoke({'document': '주민등록등본'}) == _want
assert fee_once.invoke({'document': ' 주민등록등본 '}) == _want, '앞뒤 공백을 떼세요'
_miss = fee_once.invoke({'document': '우주여권'})
assert any(k in _miss for k in FEE_TABLE), '없는 서류에는 안내 문자열을 돌려주세요'
# 반복을 권하는 문구가 설명에 남아 있으면 과다 호출이 그대로 재현된다.
assert '여러 번' not in fee_once.description, '설명에서 반복을 권하는 문장을 빼세요'
assert fee_once.description.strip(), 'docstring 을 적어 주세요'
print('✅ 통과!')

**해설**: 6번은 **구현**의 고장(인자를 안 다듬는다)이고 7번은 **설명**의 고장입니다. docstring 은 사람에게 남기는 주석이 아니라 **모델에게 주는 지시문**입니다. "확실히 하려면 여러 번" 이라고 쓰면 정말로 여러 번 부릅니다.

**과다 호출이 왜 문제인가**: 호출마다 돈과 시간이 듭니다. 그리고 도구가 부작용이 있는 것이라면(메일 발송·주문 접수) 반복은 곧 사고입니다.

**흔한 실수**: 반복을 막으려고 `system_prompt` 에 "한 번만 불러라" 를 적는 것입니다. 그것도 도움은 되지만, **원인은 도구 설명에 있으므로 거기서 고치는 것**이 맞습니다.

## 8. 루프의 단계를 스트림으로 관찰하기
**배경**: `invoke` 는 **끝난 결과**만 줍니다. 루프가 한 단계씩 진행되는 모습을 보려면 `agent.stream(..., stream_mode='values')` 로 스냅샷을 받습니다(교안 1절).

**요구사항**:
- 2번의 `civil_agent` 에 **'인감증명서 1장 수수료는?'** 을 `stream_mode='values'` 로 스트림하세요.
- 스냅샷을 돌며 개수를 세어 **`snap_count`**(정수)에 담고, **마지막 스냅샷**을 **`last_snapshot`** 에 붙잡아 두세요. 그 스냅샷의 맨 끝 메시지 타입 이름을 **`last_kind`**(문자열)에 담습니다(`type(메시지).__name__`).
- 두 값을 출력해 단계가 몇 번에 걸쳐 쌓였는지 눈으로 확인합니다.

**예시**: 도구를 한 번 부르고 답하면 `snap_count` 는 보통 **4**(질문·도구호출·도구결과·최종답), `last_kind` 는 `'AIMessage'` 입니다. 스냅샷 수는 모델이 정하는 일이라 달라질 수 있어, 자가채점은 **2 이상인지**와 마지막 스냅샷이 **실제 메시지 객체들**을 담고 있는지만 봅니다.

<details><summary>힌트</summary>

```text
접근방법:
- for 문으로 스트림을 돌며 세고, 루프가 끝난 뒤 마지막 스냅샷의 맨 끝 메시지를 본다.

세부구현:
1. snap_count = 0 으로 시작한다.
2. civil_agent.stream({'messages': [HumanMessage(질문)]}, stream_mode='values') 를 돈다.
3. 스냅샷마다 snap_count 를 1 늘리고, 그 스냅샷을 변수에 붙잡아 둔다.
4. 루프가 끝난 뒤 마지막 스냅샷의 messages 맨 끝 요소의 타입 이름을 last_kind 에 담는다.
```

</details>

In [ ]:
# 마지막 스냅샷을 변수에 담아 둔다 — 루프 밖에서 snapshot 을 그냥 쓰면 이름이 없어 막힌다
snap_count = 0
last_snapshot = None
for snapshot in civil_agent.stream({'messages': [HumanMessage('인감증명서 1장 수수료는?')]},
                                   stream_mode='values'):
    snap_count += 1
    last_snapshot = snapshot
last_kind = type(last_snapshot['messages'][-1]).__name__
print('스냅샷 수:', snap_count, '| 마지막 메시지:', last_kind)

In [ ]:
# [자가채점] - 스냅샷 수는 모델이 정하므로 '2 이상'까지만 봅니다.
from langchain_core.messages import BaseMessage
assert isinstance(snap_count, int) and snap_count >= 2
assert isinstance(last_kind, str) and last_kind.endswith('Message')
# 스트림을 실제로 돌았는지 - 마지막 스냅샷이 진짜 메시지 객체들을 담고 있어야 합니다.
assert all(isinstance(m, BaseMessage) for m in last_snapshot['messages'])
assert any(isinstance(m, AIMessage) and m.response_metadata
           for m in last_snapshot['messages'])
assert last_kind == type(last_snapshot['messages'][-1]).__name__   # 손으로 적은 이름은 여기서 어긋납니다
assert len(last_snapshot['messages']) >= snap_count
print('✅ 통과!')

**해설**: `stream_mode='values'` 는 단계마다 **그때까지 쌓인 메시지 전체**를 돌려줍니다 — 그래서 스냅샷의 맨 끝 요소가 **방금 새로 일어난 일**입니다. 첫 스냅샷은 질문만, 다음은 도구 호출(`AIMessage`), 그다음은 도구 결과(`ToolMessage`), 마지막이 최종 답(`AIMessage`)입니다. 스냅샷 수는 도구를 몇 번 부르느냐에 따라 달라지므로 정확한 수로 채점하지 않습니다. 흔한 실수: 루프 안에서 `last_kind` 를 잡지 않고 루프 뒤에 `snapshot` 을 쓰려다 이름이 없어 막히는 경우 — 마지막 스냅샷을 변수에 담아 두면 안전합니다.

---
## 9. 클라우드 벡터DB(여행지)를 도구로 감싸기
**배경**: SQL 단원에서 Supabase 에 적재한 **여행지 100곳**(`travel_docs` 표)을 오늘 배운 방식으로 감쌉니다. 검색 함수 `match_spot` 은 이미 만들어져 있으니, 여러분이 할 일은 그 결과를 **`Document` 목록**으로 바꾸고 `@tool` 로 감싸 에이전트에 붙이는 것입니다.

> **이 문제는 Supabase 접속 정보가 있어야 풀 수 있습니다.** 아래 제공 셀이 준비 상태를 확인해 `SPOT_READY`(참/거짓)에 담습니다. `⏭️` 안내가 찍혔다면 **이 문제는 건너뛰세요** — 자가채점도 함께 건너뜁니다. 풀려면 이 폴더의 `.env` 에 `SUPABASE_URL`·`SUPABASE_ANON_KEY` 를 넣고, SQL 단원에서 `travel_docs` 에 여행지를 적재해 두어야 합니다.

**요구사항** — 자가채점이 검사하는 것을 모두 적어 둡니다.

- 함수 **`find_spots(query, k=3)`** 를 만드세요.
  - `embed_spot(...)`(제공)으로 질문을 벡터로 바꾸고, `supabase.rpc` 로 **`match_spot`** 함수를 부릅니다. 인자 이름은 `query_embedding` 과 `match_count` 입니다(`match_count` 에 `k` 를 그대로 넘기세요).
  - 응답의 `.data`(딕셔너리 목록)를 **`Document` 목록**으로 바꿔 **반환**합니다.
  - `page_content` 는 **`'{name}: {description}'`**, `metadata` 에는 **`spot_id`·`region`·`area`·`spot_type`·`similarity`** 다섯 가지를 담습니다(이름 그대로).
- `@tool` 로 **`search_spot(query: str) -> str`** 을 만드세요 — `find_spots(query.strip(), 3)` 의 각 `page_content` 를 **줄바꿈으로 이어** 하나의 문자열로 돌려줍니다. docstring 에 *언제 쓰는 도구인지* 를 적으세요.
- 마지막으로 아래 두 줄을 **`if SPOT_READY:` 안에서만** 실행하세요(접속 정보가 없을 때 에러가 나지 않게).
  - `spot_agent = create_agent(model, [search_spot])`
  - `'바다가 보이는 조용한 여행지를 추천해 줘'` 를 물어 결과를 **`r_spot`** 에 담고 `tool_names(r_spot)` 출력. `SPOT_READY` 가 거짓이면 `r_spot = None` 으로 둡니다.

**예시**: `find_spots('바다가 보이는 조용한 여행지', 3)` 은 길이 **3**의 `Document` 목록이고, `tool_names(r_spot)` 에는 `'search_spot'` 이 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 6절과 같은 순서다. 질문을 벡터로 -> rpc 로 검색 -> 딕셔너리를 Document 로 -> @tool 로 감싸기.

세부구현:
1. embed_spot 으로 질문 벡터를 만든다.
2. supabase.rpc 에 함수 이름과 인자 사전을 넘기고 execute 한 뒤 .data 를 꺼낸다.
3. 각 행에서 이름과 소개글로 본문을, 나머지 다섯 값으로 꼬리표를 만들어 Document 를 쌓는다.
4. 도구는 그 목록의 본문만 줄바꿈으로 이어 문자열 하나로 만든다.
5. 에이전트를 만들어 예시 질문을 던지고, 불린 도구 이름을 출력한다.
   5-1. 이 두 줄은 SPOT_READY 가 참일 때만 실행한다.
```

</details>

In [ ]:
# [제공 코드] 여행지 벡터 표 접속 준비 - SQL 단원에서 적재한 travel_docs 를 씁니다(이 셀은 실행만 하세요).
#  접속 정보가 없거나 표가 비어 있으면 SPOT_READY 를 False 로 두고 9번만 건너뜁니다.
from supabase import create_client
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document

SPOT_READY = False
supabase = None
_spot_embedder = None
_why = ''

_url, _key = os.getenv('SUPABASE_URL'), os.getenv('SUPABASE_ANON_KEY')
if not _url or not _key:
    _why = '.env 에 SUPABASE_URL / SUPABASE_ANON_KEY 가 없습니다'
else:
    supabase = create_client(_url, _key)
    try:
        # 표가 없으면 에러가 나고, 표가 비어 있으면 에러 없이 빈 목록이 옵니다 - 둘을 따로 잡습니다.
        if supabase.table('travel_docs').select('spot_id').limit(1).execute().data:
            _spot_embedder = SentenceTransformer('jhgan/ko-sroberta-multitask')
            SPOT_READY = True
        else:
            _why = 'travel_docs 표가 비어 있습니다 - SQL 단원에서 여행지 100곳을 적재하세요'
    except Exception as _error:
        _why = f'travel_docs 표를 찾지 못했습니다 - setup_travel.sql 을 SQL Editor 에서 Run 하세요 ({_error})'


def embed_spot(text):
    """질문 한 문장을 768개 숫자 리스트로 바꾼다(적재할 때 쓴 그 모델)."""
    return _spot_embedder.encode([text], normalize_embeddings=True)[0].tolist()


print('준비 완료 - travel_docs 검색 가능' if SPOT_READY else f'⏭️ 9번은 건너뜁니다: {_why}')

In [ ]:
# rpc 응답의 .data 는 딕셔너리 목록이다 - 검색기의 규약(Document 목록)으로 바꿔 줘야 부품으로 쓰인다
def find_spots(query, k=3):
    """여행지 벡터 표에서 질문과 가까운 곳을 찾아 Document 목록으로 돌려준다."""
    rows = supabase.rpc('match_spot', {'query_embedding': embed_spot(query),
                                       'match_count': k}).execute().data
    return [Document(page_content=f"{r['name']}: {r['description']}",
                     metadata={'spot_id': r['spot_id'], 'region': r['region'],
                               'area': r['area'], 'spot_type': r['spot_type'],
                               'similarity': r['similarity']})
            for r in rows]


@tool
def search_spot(query: str) -> str:
    """가고 싶은 여행지를 말로 설명하면 어울리는 곳 세 군데를 찾아 돌려준다.

    바다·산·야경처럼 분위기나 풍경으로 여행지를 추천해 달라고 할 때 쓴다.
    """
    # 도구는 문자열을 돌려줘야 한다 - Document 목록을 그대로 주면 모델이 읽지 못한다.
    return '\n'.join(d.page_content for d in find_spots(query.strip(), 3))


# 접속 정보가 없으면 에이전트를 돌리지 않는다 - 가짜 값으로 채우지 않고 정직하게 건너뛴다.
if SPOT_READY:
    spot_agent = create_agent(model, [search_spot])
    r_spot = spot_agent.invoke({'messages': [HumanMessage('바다가 보이는 조용한 여행지를 추천해 줘')]})
    print(tool_names(r_spot), '|', r_spot['messages'][-1].text[:80])
else:
    r_spot = None
    print('⏭️ Supabase 접속 정보가 없어 에이전트 실행을 건너뜁니다.')

In [ ]:
# [자가채점] - 접속 정보가 없으면 채점하지 않고 건너뜁니다(없는 값을 지어내지 않습니다).
if not SPOT_READY:
    print('⏭️ Supabase 접속 정보(또는 travel_docs 적재)가 없어 9번 채점을 건너뜁니다.')
else:
    from langchain_core.tools import BaseTool
    # 도구·검색 함수는 모델을 거치지 않으므로 결정적으로 검사한다.
    _docs = find_spots('바다가 보이는 조용한 여행지', 3)
    assert isinstance(_docs, list) and len(_docs) == 3, '상위 3곳을 Document 목록으로 돌려주세요'
    assert all(isinstance(d, Document) for d in _docs), '딕셔너리가 아니라 Document 로 바꿔야 합니다'
    assert all(d.page_content.strip() for d in _docs), 'page_content 에 이름과 소개글을 담으세요'
    _keys = {'spot_id', 'region', 'area', 'spot_type', 'similarity'}
    assert all(_keys <= set(d.metadata) for d in _docs), f'metadata 에 {_keys} 가 모두 있어야 합니다'
    assert len(find_spots('산이 보이는 곳', 2)) == 2, 'k 를 match_count 로 그대로 넘겼는지 확인하세요'
    assert isinstance(search_spot, BaseTool) and search_spot.name == 'search_spot'
    assert isinstance(search_spot.invoke({'query': ' 바다 '}), str), '도구는 문자열을 돌려줘야 합니다'
    # 에이전트 라우팅은 구조만 본다 - 답 문장·호출 횟수는 실행마다 다르다.
    assert 'search_spot' in tool_names(r_spot), '에이전트가 search_spot 을 최소 1회 불러야 합니다'
    print('✅ 통과!')

**해설**: 교안 6절과 **같은 절차**를 다른 표(여행지)에 적용한 문제입니다. 새로 배운 것은 없습니다 — `rpc` 로 받은 딕셔너리 목록을 `Document` 목록으로 바꾸는 **규약 맞추기** 하나뿐이고, 그 뒤는 늘 하던 `@tool` + `create_agent` 입니다.

**채점을 이렇게 나눈 이유**: 검색 함수와 도구는 모델을 거치지 않아 **값이 결정적**이라 타입·개수·꼬리표까지 정확히 검사할 수 있습니다. 반대로 에이전트가 무엇을 어떻게 답하는지는 실행마다 달라지므로 **그 도구를 최소 한 번 불렀는지**라는 구조만 봅니다.

**흔한 실수**: 1) `.execute()` 를 빼먹어 응답 객체가 아니라 요청 빌더를 받는 것, 2) 도구가 `Document` 목록을 그대로 반환하는 것(도구는 문자열이어야 모델이 읽습니다), 3) `match_count` 에 `k` 를 안 넘기고 기본값 3 에 기대는 것 — `k=2` 로 불러도 3개가 와서 자가채점에서 걸립니다.

---
수고했어요! LV2 에서 **앞뒤 문맥 검색 도구·멀티툴 라우팅·필요시 검색·혼합 질문·실패 수리·스트림 관찰·클라우드 벡터DB 도구화**를 익혔습니다. LV3 에서는 이것들을 하나로 묶어 **민원 안내 에이전트**를 완성하고, 고장난 도구 셋을 진단해 고칩니다.